# Mixtures + Learned Period Head

This notebook demonstrates:
1. Selecting **top-k** periods (FFT or learned head) for each window.
2. Building **concatenated QIH histograms** for those periods.
3. Training a small classifier (if sklearn installed) for a multi-rhythm dataset.

In [ ]:
import numpy as np
from periodic_mixture_features import qih_mixture_histogram
from quantum_hybrid_system.tools_qih.learned_period_head import learned_periods_or_fft, mixture_qih_from_periods

rng = np.random.default_rng(0)

# Synthetic mixture: periods 8 and 21 with noise and occasional dropouts
def synth_mix(T=512):
    t = np.arange(T)
    x = 0.8*np.sin(2*np.pi*t/8) + 0.6*np.sin(2*np.pi*t/21) + 0.3*rng.normal(size=T)
    return x

def synth_single(T=512):
    t = np.arange(T)
    x = 1.1*np.sin(2*np.pi*t/10) + 0.3*rng.normal(size=T)
    return x

W, S = 256, 128
def windows(x):
    for start in range(0, len(x)-W+1, S):
        yield x[start:start+W]

# Build dataset of windows
Xs, Ys = [], []
for _ in range(400):
    xs = synth_mix(2048)
    for w in windows(xs):
        Xs.append(w); Ys.append(1)
for _ in range(400):
    xs = synth_single(2048)
    for w in windows(xs):
        Xs.append(w); Ys.append(0)

Xs = np.stack(Xs, axis=0)
Ys = np.array(Ys, dtype=int)
perm = rng.permutation(len(Xs)); Xs, Ys = Xs[perm], Ys[perm]

# Features
Hs = []
for w in Xs:
    periods = learned_periods_or_fft(w, model=None, topk=2)
    Hs.append(mixture_qih_from_periods(periods, bins=32))
H = np.stack(Hs, axis=0)

print("Shapes:", Xs.shape, H.shape)

try:
    from sklearn.linear_model import LogisticRegression
    from sklearn.metrics import roc_auc_score
    split = int(0.8*len(Xs))
    Htr, Hte = H[:split], H[split:]
    ytr, yte = Ys[:split], Ys[split:]
    clf = LogisticRegression(max_iter=300).fit(Htr, ytr)
    auc = roc_auc_score(yte, clf.predict_proba(Hte)[:,1])
    print({"auc_qih_mixture": float(auc)})
except Exception as e:
    print("sklearn not available:", e)